# Omnibus — interactive network map (Leaflet / folium)

`03_geo_insights.ipynb` answered the geo questions with **static** matplotlib — good for slides, useless for *exploring*. This notebook is the explorer: a real Leaflet map you can pan, zoom, click, and toggle layers on.

**What you can do here**
- Click any of the 304 geocoded stops → popup with its delay profile (median, p90, σ, lines served, sample size).
- Toggle a **σ-weighted heat layer** to see where unreliability concentrates, independent of dot size.
- Toggle **distance rings** from Hauptbahnhof (the "unreliability grows outward" story from notebook 03, Q2, made spatial).

Same caveats as the other notebooks (see [docs/DATA_DEFECTS.md](../docs/DATA_DEFECTS.md)):
- Drop `Daten_Linie_1_2024-09_2025-08` (overlaps every event window).
- Filter `|delay_arr_s| < 7200` (midnight wrap, §9).
- ~4.5% of stop-events lack coordinates (docs/GTFS.md) — filtered from the map only.

> Maps render inline in JupyterLab. GitHub won't render the embedded Leaflet iframe — open the notebook in Jupyter to interact.

In [ ]:
import polars as pl
import folium
from folium.plugins import HeatMap
import branca.colormap as cmm
import numpy as np

df = pl.read_parquet("../data/parquet/features.parquet")
ev = df.filter(pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
clean = ev.filter(pl.col("delay_arr_s").abs() < 7200)
geo = clean.filter(pl.col("stop_lat").is_not_null())

HBF_LAT, HBF_LON = 49.0125, 12.0992

# per-stop delay profile, only productive arrivals, only well-sampled stops.
# Exclude each trip's terminus (is_terminus, from assemble.py) — terminus σ is layover/
# recovery, not en-route reliability; left in, pure endpoints (Pentling, Neutraubling
# Haidauer Str.) top the σ ranking as false hotspots and stretch the colour scale.
per_stop = (
    geo.filter(pl.col("productive_arr") & ~pl.col("is_terminus"))
    .group_by("stop_name").agg(
        pl.col("delay_arr_s").median().alias("med"),
        pl.col("delay_arr_s").quantile(0.9).alias("p90"),
        pl.col("delay_arr_s").std().alias("sigma"),
        pl.col("line").unique().sort().alias("lines"),
        pl.col("stop_lat").first().alias("lat"),
        pl.col("stop_lon").first().alias("lon"),
        pl.len().alias("n"))
    .filter(pl.col("n") > 500)
    .sort("sigma", descending=True)
)
print(f"stops on map: {per_stop.height}   (rows behind them: {geo.height:,})")
per_stop.head(5).select("stop_name","med","p90","sigma","n")

## The explorer

Color = σ(arrival delay) — green predictable, red erratic. Radius ∝ √(sample size) so busy stops read bigger without swamping the map. Click for the full popup.

In [ ]:
sig = per_stop["sigma"].to_numpy()
vmin, vmax = float(np.nanpercentile(sig, 5)), float(np.nanpercentile(sig, 95))
cmap = cmm.LinearColormap(["#1a9850", "#fee08b", "#d73027"], vmin=vmin, vmax=vmax)
cmap.caption = "σ (arrival-delay std) [s] — green = predictable, red = erratic"

m = folium.Map(location=[HBF_LAT, HBF_LON], zoom_start=12, tiles="CartoDB positron")

stops_layer = folium.FeatureGroup(name="Stops (σ-colored)", show=True)
for r in per_stop.iter_rows(named=True):
    lines = ", ".join(r["lines"][:12]) + ("…" if len(r["lines"]) > 12 else "")
    popup = folium.Popup(html=(
        f"<b>{r['stop_name']}</b><br>"
        f"median delay: <b>{r['med']:.0f}s</b> &nbsp; p90: <b>{r['p90']:.0f}s</b><br>"
        f"σ: <b>{r['sigma']:.0f}s</b> &nbsp; samples: {r['n']:,}<br>"
        f"<small>lines: {lines}</small>"
    ), max_width=280)
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=3 + np.sqrt(r["n"]) / 22,
        color="#333", weight=0.4,
        fill=True, fill_color=cmap(r["sigma"]), fill_opacity=0.82,
        popup=popup,
        tooltip=f"{r['stop_name']} — σ {r['sigma']:.0f}s",
    ).add_to(stops_layer)
stops_layer.add_to(m)

# σ-weighted heat layer (toggleable)
heat = [[r["lat"], r["lon"], max(0.0, (r["sigma"] - vmin) / (vmax - vmin))]
        for r in per_stop.iter_rows(named=True)]
HeatMap(heat, name="σ heat", radius=26, blur=20, min_opacity=0.25,
        gradient={0.0: "#1a9850", 0.5: "#fee08b", 1.0: "#d73027"},
        show=False).add_to(m)

# distance rings from HBF (1/2/4/7 km)
rings = folium.FeatureGroup(name="Distance rings from HBF", show=False)
folium.Marker([HBF_LAT, HBF_LON], tooltip="Hauptbahnhof",
              icon=folium.Icon(color="blue", icon="train", prefix="fa")).add_to(rings)
for km in (1, 2, 4, 7):
    folium.Circle([HBF_LAT, HBF_LON], radius=km * 1000, color="#2c7fb8",
                  weight=1, fill=False, dash_array="4",
                  tooltip=f"{km} km").add_to(rings)
rings.add_to(m)

cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

**How to read it.** Toggle the **σ heat** layer: the northern outer ring (Wutzlhofen / Wetterstation / Harzstraße) lights up as one contiguous hot zone, with **Klinikum** standing out on its own in the south-west, while the dense Altstadt core stays cool — high traffic, but predictable. Turn on the **distance rings** and pan outward: the erratic stops cluster past the 4 km ring. That's the "unreliability grows with distance" finding (notebook 03 Q2) made tangible — you can *see* the gradient instead of reading it off a bar chart.

Each trip's terminus is excluded, so the reddest dots are real en-route chokepoints rather than layover endpoints — the former chart-toppers Pentling and Neutraubling Haidauer Str. (both trip ends) are gone, which also lets the colour scale resolve genuine variation instead of being stretched by two outliers. Click the reddest dots to confirm sample size — none of the hotspots are thin-data artifacts (all > 500 events).

## Drill into one line's route

Pick a line, order its stops by typical sequence, draw the route, and color each stop by its σ **on that line**. This is the per-line view the layer-toggle approach can't give cleanly (a stop serves many lines). Change `LINE` and re-run.

In [ ]:
LINE = "6"   # try "1", "2", "11", "8", "X4" …

ln = geo.filter(pl.col("productive_arr") & (pl.col("line") == LINE))
# order stops by their typical position along the line
route = (ln.group_by("stop_name").agg(
            pl.col("stop_seq").median().alias("seq"),
            pl.col("delay_arr_s").std().alias("sigma"),
            pl.col("delay_arr_s").median().alias("med"),
            pl.col("stop_lat").first().alias("lat"),
            pl.col("stop_lon").first().alias("lon"),
            pl.len().alias("n"))
        .filter(pl.col("n") > 50)
        .sort("seq"))

mL = folium.Map(location=[route["lat"].median(), route["lon"].median()],
                zoom_start=12, tiles="CartoDB positron")
folium.PolyLine([[r["lat"], r["lon"]] for r in route.iter_rows(named=True)],
                color="#555", weight=2, opacity=0.6).add_to(mL)
cmapL = cmm.LinearColormap(["#1a9850", "#fee08b", "#d73027"],
                           vmin=float(route["sigma"].min()),
                           vmax=float(route["sigma"].max()))
cmapL.caption = f"Line {LINE} — σ(arrival delay) [s] along the route"
for i, r in enumerate(route.iter_rows(named=True)):
    folium.CircleMarker(
        [r["lat"], r["lon"]], radius=6, color="#222", weight=0.5,
        fill=True, fill_color=cmapL(r["sigma"]), fill_opacity=0.9,
        tooltip=f"#{i+1} {r['stop_name']} — σ {r['sigma']:.0f}s, med {r['med']:.0f}s",
    ).add_to(mL)
cmapL.add_to(mL)
print(f"Line {LINE}: {route.height} stops along the route")
mL

**How to read it.** The route is laid out in service order, so you can watch σ build up as you trace from terminus to terminus — green at the start, reddening downstream. Where a green run suddenly jumps to red marks the segment that *injects* variance (a chronic chokepoint), as opposed to stops that are merely red because they inherited delay from upstream. That distinction is exactly what Scene A needs and what the static maps blur together.

> Branching lines (median-seq ordering) can draw a small zig-zag where two branches share a number — harmless for reading σ per stop; ignore the connecting line there.